### *Uncovering the multiple socio-economic driving factors of carbon emissions in nine urban agglomerations of China*

> [Demystifying the XGBoost Black Box with a SHAP & Nightingale Rose Combination Plot in Python](https://medium.com/top-python-libraries/journal-reproduction-demystifying-the-xgboost-black-box-with-a-shap-nightingale-rose-21d18b727799)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

#### Environment preparation and global font settings

In [ ]:
import pandas as pd
import numpy as np
import xgboost
import shap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable

import matplotlib
matplotlib.use('TkAgg')

from sklearn.model_selection import GridSearchCV

plt.style.use('ggplot')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.linewidth'] = 3
plt.rcParams['xtick.major.width'] = 3
plt.rcParams['ytick.major.width'] = 3
plt.rcParams['xtick.minor.width'] = 3.0
plt.rcParams['ytick.minor.width'] = 1.0

#### Data loading and feature separation

In [ ]:
# Load the data
excel_path = r'https://github.com/checkming00/Medium_datasets/raw/refs/heads/main/simulated_data.xlsx'
data_df = pd.read_excel(excel_path)
print(f"Successfully loaded data from '{excel_path}'.")

# Define the target variable
target_column_name = 'Target_Variable'

# Check if the target column exists in the data
if target_column_name not in data_df.columns:
    print(f"Error: The specified target column '{target_column_name}' does not exist in the Excel file!")
    print(f"The columns included in the file are: {data_df.columns.tolist()}")

# Separate features (X) and target (y) based on the target column name
y = data_df[target_column_name]
X = data_df.drop(columns=[target_column_name])

# Automatically get the list of feature names from the loaded data
feature_names = X.columns.tolist()
print(f"Identified target variable: '{target_column_name}'")
print(f"Identified {len(feature_names)} feature variables.")

#### XGBoost model training and hyperparameter optimization

In [ ]:
# Build and train the XGBoost model
# ------------------------------------

# Define the hyperparameter grid to search
param_grid = {
    'n_estimators': [100, 200],      # Number of trees
    # 'max_depth': [3, 5, 7],          # Maximum depth of a tree
    # 'learning_rate': [0.05, 0.1, 0.2], # Learning rate
    # 'subsample': [0.8, 1.0]           # Subsample ratio of the training instance for each tree
}

# Initialize an XGBoost regressor model
xgb_reg = xgboost.XGBRegressor(objective='reg:squarederror', random_state=42)

# Set up GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(estimator=xgb_reg, param_grid=param_grid,
                           cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)
print("\nStarting hyperparameter search and cross-validation...")

# Use the simulated data to perform the grid search
grid_search.fit(X, y)

# Get the best model found by the search
best_model = grid_search.best_estimator_
print("Hyperparameter search complete.")
print(f"Best parameters found: {grid_search.best_params_}")
print("Model has been set with the best parameters and is ready for SHAP analysis.")

#### SHAP analysis and feature importance ranking

In [ ]:
# Perform SHAP analysis using the best model
model = best_model

# Create a TreeExplainer object, used for explaining tree-based models
explainer = shap.TreeExplainer(model)

# Calculate the SHAP values for each feature of each sample in the dataset
shap_values = explainer(X)

# Calculate the mean absolute SHAP value for each feature as a measure of feature importance
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

# Combine the mean absolute SHAP values and feature names into a pandas Series
shap_series = pd.Series(mean_abs_shap, index=X.columns)

# Reindex the Series according to the original order of feature_names
shap_series = shap_series.reindex(feature_names)

# Sort the features in descending order by SHAP value
shap_series.sort_values(ascending=False, inplace=True)

# Print that the SHAP analysis and sorting are complete
print("SHAP analysis complete, feature importances have been sorted.")

#### Feature Importance Bar Chart with Inline Radial Plot

In [ ]:
# Create a 16x15 inch figure
fig = plt.figure(figsize=(16, 15))

# Define the figure margins
left_margin, right_margin, bottom_margin, top_margin = 0.08, 0.08, 0.12, 0.12

# Define the space between plots
space_between_plots = 0.04

# Define the width of the color bar
colorbar_width = 0.02

# Define the bottom position of the main plot
plot_bottom = bottom_margin

# Define the height of the main plot
plot_height = 1.0 - bottom_margin - top_margin

# Define the left position of the color bar
cbar_left = left_margin

# Define the left position of the main axes
main_ax_left = cbar_left + colorbar_width + space_between_plots

# Define the width of the main axes
main_ax_width = 1.0 - main_ax_left - right_margin

# Add axes for the color bar to the figure
ax_cbar = fig.add_axes([cbar_left, plot_bottom, colorbar_width, plot_height])

# Add axes for the main bar plot to the figure
ax_bar = fig.add_axes([main_ax_left, plot_bottom, main_ax_width, plot_height])

# Create a ScalarMappable object to map data values to colors
sm = ScalarMappable(cmap=cmap, norm=color_norm)

# Draw the color bar on the specified axes (ax_cbar)
cbar = fig.colorbar(sm, cax=ax_cbar, orientation='vertical')

# Set the color bar's label to empty, but reserve space
cbar.set_label('', size=18, labelpad=15)

# Remove the ticks from the color bar
cbar.set_ticks([])

# Set the color bar tick position to the left
cbar.ax.yaxis.set_ticks_position('left')

# Add 'High' text at the top of the color bar
ax_cbar.text(0.5, 1.01, 'High', transform=ax_cbar.transAxes, ha='center', va='bottom', fontsize=24)

# Add 'Low' text at the bottom of the color bar
ax_cbar.text(0.5, -0.01, 'Low', transform=ax_cbar.transAxes, ha='center', va='top', fontsize=24)

# Hide the color bar's outline
cbar.outline.set_visible(False)

# Add a rotated text label 'Contribution for CEs ($10^4$ t)' to the left of the color bar
ax_cbar.text(-1.4, 0.5, 'Contribution for CEs ($10^4$ t)', transform=ax_cbar.transAxes, fontsize=24, rotation=90, va='center')

# Set the main plot's X-axis ticks to the bottom
ax_bar.xaxis.tick_bottom()

# Set the main plot's X-axis label position to the bottom
ax_bar.xaxis.set_label_position("bottom")

# Invert the main plot's X-axis direction
ax_bar.invert_xaxis()

# Draw the horizontal bar plot
ax_bar.barh(y=range(len(sorted_features)), width=sorted_shap_values, color=colors, height=0.6)

# Invert the main plot's Y-axis so the most important feature is at the top
ax_bar.invert_yaxis()

# Set the main plot's X-axis label
ax_bar.set_xlabel('Contribution for CEs ($10^4$ t)', size=24, labelpad=20)

# Remove the main plot's Y-axis ticks
ax_bar.set_yticks([])

# Hide the left and top spines of the axes
ax_bar.spines[['left', 'top']].set_visible(False)

# Move the right spine to the data position 0
ax_bar.spines['right'].set_position(('data', 0))

# Make the right spine visible
ax_bar.spines['right'].set_visible(True)

# Make the bottom spine visible
ax_bar.spines['bottom'].set_visible(True)

# Set the style for the X-axis major ticks
ax_bar.tick_params(axis='x', which='major', direction='in', labelsize=24, length=6, pad=8)

# Automatically set the minor locator for the X-axis
ax_bar.xaxis.set_minor_locator(ticker.AutoMinorLocator(10))

# Set the style for the X-axis minor ticks
ax_bar.tick_params(axis='x', which='minor', direction='in', length=4)

# Define the padding for feature labels in the X direction
label_x_padding = 0.005

# Iterate through the sorted features and add text labels to the plot
for i, feature in enumerate(sorted_features):
    ax_bar.text(label_x_padding, i, feature, ha='right', va='center', color='black', fontsize=24)

# Add the subplot label '(a)' to the top-left corner of the main plot
ax_bar.text(0.02, 0.98, '(a)', transform=ax_bar.transAxes, fontsize=30, weight='bold', ha='left', va='top')

# Calculate the left position of the radial plot (inset)
inset_left = main_ax_left - 0.15

# Calculate the bottom position of the radial plot
inset_bottom = plot_bottom - 0.05

# Calculate the size of the radial plot
inset_size = min(main_ax_width, plot_height) * 0.85

# Define the position and size of the radial plot [left, bottom, width, height]
inset_ax_rect = [inset_left, inset_bottom, inset_size, inset_size]

# Add axes for the radial plot to the figure, and set it to a polar projection
ax_radial_inset = fig.add_axes(inset_ax_rect, projection='polar')

# Set the radial plot background to transparent
ax_radial_inset.patch.set_alpha(0)

# Calculate the percentage of the total SHAP value for each feature
percentages = (sorted_shap_values / sorted_shap_values.sum()) * 100

# Calculate the width (in radians) of each sector based on the percentage
widths = (sorted_shap_values / sorted_shap_values.sum()) * 2 * np.pi

# Get the number of features
num_vars = len(sorted_features)

# Define the base length, fixed increment, and colored ring width for the radial plot
base_length, fixed_increment, colored_ring_width = 3.0, 0.5, 2.0

# Calculate the total length of each sector
total_lengths = [base_length + i * fixed_increment for i in range(num_vars)]

# Calculate the height of the inner gray/white rings
inner_heights = [max(0, tl - colored_ring_width) for tl in total_lengths]

# Define the alternating colors for the inner rings
inner_colors = ['#EAEAEA', '#FFFFFF'] * (num_vars // 2 + 1)

# Slice the required number of colors
inner_colors = inner_colors[:num_vars]

# Set an offset to start the first sector near the 1 o'clock position
one_oclock_offset = np.pi / 21

# Calculate the starting angle (theta) for each sector
thetas = np.cumsum([0] + widths[:-1].tolist()) - one_oclock_offset

# Draw the inner alternating gray/white rings
ax_radial_inset.bar(x=thetas, height=inner_heights, width=widths, color=inner_colors, align='edge', edgecolor='white', linewidth=1.5)

# Draw the outer colored rings
ax_radial_inset.bar(x=thetas, height=[colored_ring_width] * num_vars, width=widths, bottom=inner_heights, color=colors, align='edge', edgecolor='white',
linewidth=1.5)

# Iterate through each sector and add percentage labels
for i in range(num_vars):
    # Calculate the angle for the label
    label_angle_rad = thetas[i] + widths[i] / 2

    # Calculate the radius for the label
    label_radius = total_lengths[i] + 0.5

    # Add text at the specified position
    ax_radial_inset.text(label_angle_rad, label_radius, f'{percentages[i]:.1f}%', ha='center', va='center', fontsize=18)

# Remove the Y-axis (radial) labels of the radial plot
ax_radial_inset.set_yticklabels([])

# Remove the X-axis (angular) labels of the radial plot
ax_radial_inset.set_xticklabels([])

# Hide the outer circle of the polar plot
ax_radial_inset.spines['polar'].set_visible(False)

# Hide the grid lines
ax_radial_inset.grid(False)

# Set the zero-degree angle (theta=0) to the North direction
ax_radial_inset.set_theta_zero_location('N')

# Set the direction of angle increase to clockwise
ax_radial_inset.set_theta_direction(-1)

# Set the range of the Y-axis (radius)
ax_radial_inset.set_ylim(0, max(total_lengths) + 2)

# Define the save path for the original combined plot
original_image_path = r'/content/shap.png'

# Save the figure, set the DPI and crop whitespace
plt.savefig(original_image_path, dpi=208, bbox_inches='tight')

# Display the plot
plt.show()

#### SHAP honeycomb summary chart

In [ ]:
# Print a message indicating that the beeswarm plot is being drawn
print("\nDrawing beeswarm summary plot using the `shap.summary_plot` function...")

# ------------------ Version 1: Native beeswarm plot with feature labels ------------------
plt.figure(figsize=(16, 15))

# Directly call the summary_plot function
shap.summary_plot(shap_values, X, plot_type="dot", show=False, cmap=cmap)

# Fine-tune the plot
ax = plt.gca()
ax.set_xlabel("SHAP Value (impact on model output)", fontsize=18)
ax.tick_params(axis='y', labelsize=16)
ax.tick_params(axis='x', labelsize=14)

# Adjust the color bar if it exists
if len(plt.gcf().axes) > 1:
    cbar_ax = plt.gcf().axes[-1]
    cbar_ax.set_ylabel('Feature Value', size=16, rotation=-90, labelpad=20)
    cbar_ax.tick_params(labelsize=14)
plt.tight_layout()

# Define the save path for the beeswarm plot
beeswarm_image_path = r'D:\shap_beeswarm_native.png'

# Save the figure
plt.savefig(beeswarm_image_path, dpi=208, bbox_inches='tight')

# Display the plot
plt.show()

# ------------------ Version 2: Beeswarm plot without Y-axis feature labels ------------------
plt.figure(figsize=(16, 15))

# Call the summary_plot function again to generate the plot content
shap.summary_plot(shap_values, X, plot_type="dot", show=False, cmap=cmap)

# Get the axes and modify them
ax_third_plot = plt.gca()

# Remove the Y-axis tick labels and axis title
ax_third_plot.set_yticklabels([])
ax_third_plot.set_ylabel('')

# Add the (b) annotation
ax_third_plot.text(1, 0.98, '(b)',
                   transform=ax_third_plot.transAxes,
                   fontsize=24,
                   fontweight='bold',
                   va='top',
                   ha='right')

# Set the X-axis label and font size
ax_third_plot.set_xlabel("SHAP Value (impact on model output)", fontsize=18)
ax_third_plot.tick_params(axis='x', labelsize=14)

# Adjust the color bar
if len(plt.gcf().axes) > 1:
    cbar_ax_third = plt.gcf().axes[-1]
    cbar_ax_third.set_ylabel('Feature Value', size=16, rotation=-90, labelpad=20)
    cbar_ax_third.tick_params(labelsize=14)
plt.tight_layout()

# Define the save path for the plot without labels
third_plot_image_path = r'D:\shap_beeswarm_no_labels.png'

# Save the figure
plt.savefig(third_plot_image_path, dpi=208, bbox_inches='tight')

# Display the plot
plt.show()

#### Group pictures

In [ ]:
# ===================================================================
# Merge plot (a) and plot (b)
# ===================================================================

# At the top of the script, please ensure this import statement is present
from shap.plots import beeswarm # Directly import the beeswarm function from the shap.plots module
print("\nCombining and drawing plot (a) and plot (b) into a single figure (preserving original styles)...") # Print a message for the current operation

# 1. Create a wider new figure to provide ample space for the original styles
fig_combined = plt.figure(figsize=(34, 25)) # Create a figure object with a width of 34 inches and a height of 25 inches

# 2. Define overall layout parameters to allocate reasonable space for the two plots
# Global layout
left_margin = 0.05      # Define the left margin of the entire figure as 5% of the figure width
right_margin = 0.05     # Define the right margin of the entire figure as 5% of the figure width
bottom_margin = 0.12    # Define the bottom margin of the entire figure as 12% of the figure height
top_margin = 0.1        # Define the top margin of the entire figure as 10% of the figure height
space_between = 0.01    # Define the horizontal gap between the two subplots as 1% of the figure width

# Calculate the dimensions of the plotting area
plot_bottom = bottom_margin # Set the bottom position of the plotting area
plot_height = 1 - bottom_margin - top_margin # Calculate the actual height of the plotting area
total_plot_width = 1 - left_margin - right_margin - space_between # Calculate the total width available for plotting
left_plot_width = total_plot_width * 0.6  # Allocate 60% of the total plotting width to the left plot
right_plot_width = total_plot_width * 0.4 # Allocate 40% of the total plotting width to the right plot

# =======================================================
# Draw the left plot (plot a)
# =======================================================

# --- Color bar for the left plot ---
cbar_left = 0.1# Set the starting left position of the color bar
colorbar_width = 0.01 # Set the width of the color bar
ax_cbar_new = fig_combined.add_axes([cbar_left, plot_bottom, colorbar_width, plot_height]) # Add axes for the color bar to the figure
sm = ScalarMappable(cmap=cmap, norm=color_norm) # Create a ScalarMappable object to map data values to colors
cbar = fig_combined.colorbar(sm, cax=ax_cbar_new, orientation='vertical') # Draw a vertical color bar on the specified axes
cbar.set_label('', size=18, labelpad=5) # Set the color bar label to empty, but keep font size and padding settings
cbar.set_ticks([]) # Remove all ticks from the color bar
cbar.ax.yaxis.set_ticks_position('left') # Set the color bar's tick position (even if invisible) to the left
ax_cbar_new.text(0.5, 1.01, 'High', transform=ax_cbar_new.transAxes, ha='center', va='bottom', fontsize=30) # Add 'High' text at the top of the color bar
ax_cbar_new.text(0.5, -0.01, 'Low', transform=ax_cbar_new.transAxes, ha='center', va='top', fontsize=30) # Add 'Low' text at the bottom of the color bar
cbar.outline.set_visible(False) # Hide the color bar's outline border
ax_cbar_new.text(-1.4, 0.5, 'Contribution for CEs ($10^4$ t)', transform=ax_cbar_new.transAxes, fontsize=30, rotation=90, va='center') # Add a 90-degree rotated text label to the left of the color bar

# --- Main bar plot for the left plot ---
main_ax_left = cbar_left + colorbar_width + 0.05 # Calculate the starting left position of the main bar plot
ax_bar_new = fig_combined.add_axes([main_ax_left, plot_bottom, left_plot_width, plot_height]) # Add axes for the main bar plot to the figure
ax_bar_new.xaxis.tick_bottom() # Set the X-axis ticks to the bottom of the axes
ax_bar_new.xaxis.set_label_position("bottom") # Set the X-axis label to the bottom of the axes
ax_bar_new.invert_xaxis() # Invert the X-axis direction, so the bars extend from right to left
ax_bar_new.barh(y=range(len(sorted_features)), width=sorted_shap_values, color=colors, height=0.6) # Draw a horizontal bar plot
ax_bar_new.invert_yaxis() # Invert the Y-axis direction to display the most important feature at the top
ax_bar_new.set_xlabel('Contribution for CEs ($10^4$ t)', size=30, labelpad=20) # Set the X-axis label, its font size, and padding
ax_bar_new.set_yticks([]) # Remove all Y-axis ticks
ax_bar_new.spines[['left', 'top']].set_visible(False) # Hide the left and top spines (borders) of the axes
ax_bar_new.spines['right'].set_position(('data', 0)) # Move the right spine to the position where X-axis data is 0
ax_bar_new.spines['right'].set_visible(True) # Make the right spine visible
ax_bar_new.spines['bottom'].set_visible(True) # Make the bottom spine visible
ax_bar_new.tick_params(axis='x', which='major', direction='in', labelsize=30, length=6, pad=8) # Set the style for the X-axis major ticks
ax_bar_new.xaxis.set_minor_locator(ticker.AutoMinorLocator(10)) # Automatically set the minor locator for the X-axis
ax_bar_new.tick_params(axis='x', which='minor', direction='in', length=4) # Set the style for the X-axis minor ticks
label_x_padding = 0.005 # Padding value for the X-axis position of text labels
for i, feature in enumerate(sorted_features): # Iterate through each of the sorted features
    # Right-align the text to the axes
    ax_bar_new.text(label_x_padding, i, feature, ha='right', va='center', color='black', fontsize=30) # Add the feature name text to the plot and right-align it
ax_bar_new.text(0.02, 0.98, '(a)', transform=ax_bar_new.transAxes, fontsize=30, weight='bold', ha='left', va='top') # Add the subplot label '(a)' to the top-left corner of the main plot

# --- Inset radial plot for the left plot ---
inset_size = min(left_plot_width, plot_height) * 0.85 # Calculate the size of the inset radial plot
inset_left = main_ax_left - 0.15 # Calculate the left position of the inset plot
inset_bottom = plot_bottom - 0.05 # Calculate the bottom position of the inset plot
inset_ax_rect = [inset_left, inset_bottom, inset_size, inset_size] # Define the position and size of the inset plot
ax_radial_inset_new = fig_combined.add_axes(inset_ax_rect, projection='polar') # Add an inset axes with a polar projection to the figure
ax_radial_inset_new.patch.set_alpha(0) # Set the background of the inset plot to be fully transparent
ax_radial_inset_new.bar(x=thetas, height=inner_heights, width=widths, color=inner_colors, align='edge', edgecolor='white', linewidth=1.5) # Draw the inner alternating gray/white rings
ax_radial_inset_new.bar(x=thetas, height=[colored_ring_width] * num_vars, width=widths, bottom=inner_heights, color=colors, align='edge', edgecolor='white', linewidth=1.5) # Draw the outer colored rings
for i in range(num_vars): # Iterate through each feature sector
    label_angle_rad = thetas[i] + widths[i] / 2 # Calculate the angle for the percentage label
    label_radius = total_lengths[i] + 0.5 # Calculate the radius for the percentage label
    ax_radial_inset_new.text(label_angle_rad, label_radius, f'{percentages[i]:.1f}%', ha='center', va='center', fontsize=26) # Add the percentage text label outside the sector
ax_radial_inset_new.set_yticklabels([]) # Remove the Y-axis (radial) tick labels of the radial plot
ax_radial_inset_new.set_xticklabels([]) # Remove the X-axis (angular) tick labels of the radial plot
ax_radial_inset_new.spines['polar'].set_visible(False) # Hide the outer circle of the polar plot
ax_radial_inset_new.grid(False) # Hide the polar grid lines
ax_radial_inset_new.set_theta_zero_location('N') # Set the zero-degree angle (theta=0) to the North direction
ax_radial_inset_new.set_theta_direction(-1) # Set the direction of angle increase to clockwise
ax_radial_inset_new.set_ylim(0, max(total_lengths) + 2) # Set the range of the Y-axis (radius)

# =======================================================
# Draw the right plot (plot b)
# =======================================================

# Create new axes for the SHAP beeswarm plot on the right
right_plot_left = main_ax_left + left_plot_width + space_between # Calculate the starting left position of the right plot
ax_beeswarm = fig_combined.add_axes([right_plot_left, plot_bottom, right_plot_width, plot_height]) # Add axes for the beeswarm plot to the figure

# Use the beeswarm function to draw the plot on the specified axes
beeswarm(shap_values, max_display=len(sorted_features),ax=ax_beeswarm, show=False, color=cmap, plot_size=None) # Call the beeswarm function to plot, specifying the axes, colormap, and disabling its automatic size adjustment

# Customize the right plot
ax_beeswarm.set_yticklabels([]) # Remove all Y-axis tick labels
ax_beeswarm.set_ylabel('') # Remove the Y-axis title
ax_beeswarm.set_xlabel("SHAP Value (impact on model output)", fontsize=30) # Set the X-axis label and its font size
ax_beeswarm.tick_params(axis='x', labelsize=30) # Set the font size for the X-axis tick labels

# Add the label '(b)' to the top-right corner
ax_beeswarm.text(0.98, 0.98, '(b)', # Add text at the relative position (0.98, 0.98) within the axes
                 transform=ax_beeswarm.transAxes, # Specify the coordinate system as the axes' relative coordinate system
                 fontsize=30, # Set the font size
                 fontweight='bold', # Set the font weight to bold
                 va='top', # Set the vertical alignment to top
                 ha='right') # Set the horizontal alignment to right

# Adjust the auto-generated color bar for the right plot
if len(fig_combined.axes) > 3: # Check if a color bar was auto-generated by the beeswarm plot
    cbar_ax_right = fig_combined.axes[-1] # Get the last axes, which is the color bar axes for the beeswarm plot
    cbar_ax_right.set_ylabel('Feature Value', size=30, rotation=270, labelpad=5) # Set the color bar's label, font size, rotation angle, and padding
    cbar_ax_right.tick_params(labelsize=30) # Set the font size for the color bar's tick labels

# =======================================================
# Save and display the final combined plot
# =======================================================

combined_image_path = r'D:\combined_shap_plot_final_style_preserved.png' # Define the save path and filename for the final combined plot
plt.savefig(combined_image_path, dpi=208, bbox_inches='tight') # Save the figure, setting the resolution (dpi) and cropping the excess white space around it
plt.show() # Display the plotted figure